# TruthBomb — Claim Extractor Fine-Tuning

Fine-tunes **Phi-3.5-mini-instruct** into the TruthBomb *claim extractor*: it takes a raw, code-mixed Sri Lankan social-media post (Singlish / Sinhala / Tamil / English) and emits a single **normalized English declarative claim** ready for fact-checking.

**This notebook produces two deliberately different artifacts:**

| Artifact | Format | Used by |
|---|---|---|
| `phi3-extractor-lora.zip` | LoRA adapter | The agentic **Local Researcher** (base Phi-3 + adapter at runtime) |
| `phi3-extractor-unsloth.Q4_K_M.gguf` | 4-bit GGUF | The deterministic extractor (`llama-cpp-python`, `temp=0`, GBNF) |

**Run order:** the cells are sequential and assume a **CUDA GPU runtime** (Colab T4 works; A100/L4 is faster). Steps: install → fetch data → fine-tune → export adapter → merge → build quantizer → convert to GGUF → evaluate.

## 1 · Environment Setup

Installs Unsloth and its dependencies. The install branches on GPU compute capability: newer cards (≥ 8.0: A100/L4) additionally get `flash-attn`, which the free-tier T4 cannot use. `--no-deps` prevents these packages from overwriting Colab's pre-installed CUDA stack.

In [1]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# Install Unsloth's base Colab package
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# GPU-architecture-specific deps, installed WITHOUT recompiling (--no-deps)
if major_version >= 8:
    # A100 / L4 / newer: flash-attention is supported
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    # Tesla T4 (free Colab GPU): no flash-attn
    !pip install --no-deps xformers trl peft accelerate bitsandbytes

!pip install datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-qd2_sk9h/unsloth_415ae47a82714232a63900da117ead2a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-qd2_sk9h/unsloth_415ae47a82714232a63900da117ead2a
  Resolved https://github.com/unslothai/unsloth.git to commit d0f8d40c36709447595acad3b3f4468fc8b0edaf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 21.8 MB/s eta 0:00:00
  

## 2 · Dataset

Downloads `extraction_train.jsonl`, the supervised training set. Each row is **Alpaca-format** with `instruction` / `input` (raw post) / `output` (gold normalized claim). The model learns the *transformation* from messy post to clean claim from these pairs.

In [ ]:
# Download the claim-extraction dataset (Alpaca-format JSONL) from Google Drive
# !wget --no-check-certificate -O extraction_train.jsonl "https://drive.google.com/uc?export=download&id=1mdiIfYqdU9R1WuO5mwb8PUjdk9UTrMjH"

--2026-06-30 16:43:11--  https://drive.google.com/uc?export=download&id=1mdiIfYqdU9R1WuO5mwb8PUjdk9UTrMjH
Resolving drive.google.com (drive.google.com)... 142.251.2.100, 142.251.2.113, 142.251.2.139, ...
Connecting to drive.google.com (drive.google.com)|142.251.2.100|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1mdiIfYqdU9R1WuO5mwb8PUjdk9UTrMjH&export=download [following]
--2026-06-30 16:43:11--  https://drive.usercontent.google.com/download?id=1mdiIfYqdU9R1WuO5mwb8PUjdk9UTrMjH&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.141.132, 2607:f8b0:4023:c0b::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.141.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 52198 (51K) [application/octet-stream]
Saving to: ‘extraction_train.jsonl’

extraction_train.js 100%[===================>]  50.97K  --.-K

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Data set breakdown into two files

In [7]:
import os, json, random, shutil
from collections import defaultdict, Counter

SRC_FULL      = "drive/MyDrive/Advance_AI_Files/extraction_full.jsonl"   # preserved copy of the ORIGINAL dataset
TRAIN_OUT     = "extraction_train.jsonl"  # 75% -> consumed by the fine-tuning cell
EVAL_OUT      = "extraction_eval.jsonl"   # 25% -> consumed by the evaluation cell
EVAL_FRACTION = 0.25                       # 3 : 1  (train : eval)
SEED          = 3407

# Preserve the original once so this cell is idempotent.
# (If you ever swap in a new dataset, delete extraction_full.jsonl to re-bootstrap.)
if not os.path.exists(SRC_FULL):
    shutil.copy(TRAIN_OUT, SRC_FULL)

rows = [json.loads(l) for l in open(SRC_FULL, encoding="utf-8") if l.strip()]
random.seed(SEED)

# Group by gold claim so identical claims never straddle the split (no leakage).
groups = defaultdict(list)
for r in rows:
    groups[r["output"]].append(r)

def style_of(key):
    g = groups[key][0]
    return g.get("style") or g.get("language") or "all"

# Per-style eval quotas (proportional). Falls back to one "all" bucket if no style field.
keys = list(groups)
random.shuffle(keys)
target_eval = round(len(rows) * EVAL_FRACTION)
style_total = Counter(style_of(k) for k in keys)
style_quota = {s: round(target_eval * c / len(rows)) for s, c in style_total.items()}

eval_rows, train_rows, filled = [], [], Counter()
for k in keys:
    s = style_of(k)
    if len(eval_rows) < target_eval and filled[s] < style_quota.get(s, 0):
        eval_rows += groups[k]; filled[s] += len(groups[k])
    else:
        train_rows += groups[k]

random.shuffle(train_rows); random.shuffle(eval_rows)

for path, data in [(TRAIN_OUT, train_rows), (EVAL_OUT, eval_rows)]:
    with open(path, "w", encoding="utf-8") as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ---- report ----
leak = {r["output"] for r in train_rows} & {r["output"] for r in eval_rows}
print(f"source rows : {len(rows)}")
print(f"train       : {len(train_rows)}  ({len(train_rows)/len(rows)*100:.0f}%)")
print(f"eval        : {len(eval_rows)}  ({len(eval_rows)/len(rows)*100:.0f}%)")
print(f"claim leakage (must be 0): {len(leak)}")
print("eval by style :", dict(Counter(style_of_r for r in eval_rows
      for style_of_r in [r.get('style') or r.get('language') or 'all'])))

source rows : 379
train       : 336  (89%)
eval        : 43  (11%)
claim leakage (must be 0): 0
eval by style : {'all': 43}


## 3 · Fine-Tuning (QLoRA)

Loads Phi-3.5-mini in 4-bit and attaches **LoRA** adapters (`r=16`, `alpha=16`) to all attention and MLP projections — only these small adapters are trained, the base weights stay frozen.

Key details:
- **Alpaca template** wraps every example. This exact template **must be reused at inference** — a mismatch is the most common silent accuracy killer.
- **`EOS_TOKEN`** is appended to each target so the model learns where to stop (critical for clean, single-claim output under deterministic decoding).
- **Effective batch size 8** (`2 × 4` gradient accumulation), `3` epochs, LR `2e-4` linear, `adamw_8bit`, seed `3407` for reproducibility.
- Uses the current `trl` API where `dataset_text_field`, `max_seq_length`, etc. live inside **`SFTConfig`**.

Saves the trained LoRA adapter to `phi3-extractor-lora/`.

In [8]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# Configuration
max_seq_length = 1024
dataset_file = TRAIN_OUT
output_adapter_dir = "phi3-extractor-lora"

# Load base model in 4-bit (QLoRA)
print("Loading pre-trained Phi-3.5-mini-instruct in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Attach LoRA adapters to all attention + MLP projections
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # large VRAM saving
    random_state = 3407,
)

# Alpaca prompt template - MUST be identical at inference time
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token  # appended so the model learns where to stop

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction=instruction, input=input, output=output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

print("Processing local JSONL dataset...")
dataset = load_dataset("json", data_files={"train": dataset_file}, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# SFT trainer (current trl API: dataset/seq args live in SFTConfig)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch size = 8
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",              # quantized optimizer states, saves VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training pass...")
trainer_stats = trainer.train()
print(f"Training complete! Final Loss: {trainer_stats.metrics['train_loss']}")

# Save the LoRA adapters
model.save_pretrained(output_adapter_dir)
tokenizer.save_pretrained(output_adapter_dir)
print(f"Adapters saved to: '{output_adapter_dir}'")

Loading pre-trained Phi-3.5-mini-instruct in 4-bit...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing local JSONL dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/336 [00:00<?, ? examples/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-126.


Training complete! Final Loss: 7.386806245834109


Unsloth: Restored added_tokens_decoder metadata in phi3-extractor-lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3-extractor-lora.


Adapters saved to: 'phi3-extractor-lora'


## 4 · Export the LoRA Adapter

Zips and downloads the adapter. This is the artifact the **Local Researcher agent** loads on top of base Phi-3 at runtime (`PeftModel` + adapter path).

In [ ]:
import shutil
from google.colab import files

# Zip the adapter folder and download it
shutil.make_archive("phi3-extractor-lora", "zip", "phi3-extractor-lora")
files.download("phi3-extractor-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5 · Merge to 16-bit

Fuses the LoRA weights back into the base model to produce a single standalone checkpoint. This is **not** redundant with Step 4 — GGUF conversion requires a merged checkpoint, not separated adapters.

In [9]:
# Merge LoRA weights into the base model (required before GGUF conversion)
model.save_pretrained_merged("phi3-extractor-16bit", tokenizer, save_method="merged_16bit")

config.json:   0%|          | 0.00/3.31k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in phi3-extractor-16bit/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3-extractor-16bit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [03:35<03:35, 215.83s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [05:08<00:00, 154.22s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:57<00:00, 88.85s/it]


Unsloth: Merge process complete. Saved to `/content/phi3-extractor-16bit`


## 6 · Build the llama.cpp Quantizer

Clones llama.cpp and compiles **only** the `llama-quantize` target (with CUDA). A minimal build — the full inference stack is not needed here, just the quantizer used in the next step.

In [ ]:
# Build dependencies
!apt-get update && apt-get install build-essential cmake xxd -y

# Clone llama.cpp
!git clone https://github.com/ggml-org/llama.cpp

# Configure and compile only the quantizer target (CUDA on)
!cmake llama.cpp -B llama.cpp/build -DGGML_CUDA=ON
!cmake --build llama.cpp/build --config Release --target llama-quantize

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,800 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,258 kB]
Hit:13 https://ppa.launchpadcontent.n

## 7 · Convert and Quantize to GGUF

Two steps: convert the merged 16-bit checkpoint to an uncompressed **f16 GGUF**, then quantize it down to **Q4_K_M** — the small, fast, 4-bit format that powers the deterministic extractor in production. The final cell downloads the deployable `.gguf`.

In [ ]:
# Convert the merged 16-bit checkpoint to an uncompressed f16 GGUF
!python llama.cpp/convert_hf_to_gguf.py phi3-extractor-16bit --outfile phi3-extractor-f16.gguf
print("Conversion done")

# Quantize down to the deployable 4-bit Q4_K_M layout
!./llama.cpp/build/bin/llama-quantize phi3-extractor-f16.gguf phi3-extractor-unsloth.Q4_K_M.gguf Q4_K_M

INFO:hf-to-gguf:Loading model: phi3-extractor-16bit
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:hf-to-gguf:heuristics detected bfloat16 tensor dtype, setting --outtype bf16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> BF16, shape = {3072, 32064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> BF16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> BF16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --

In [ ]:
from google.colab import files
files.download("phi3-extractor-unsloth.Q4_K_M.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8 · Evaluation Harness

Measures extraction quality against a **held-out** set using a layered metric suite — exact match, token-F1, ROUGE-L, **chrF** (best for code-mixed Singlish), and **semantic cosine** (the headline metric, via the same multilingual embedder the pipeline uses). Results are broken down **per language style** to expose Singlish-specific weakness.

> **Prerequisite — not yet present.** This cell reads `EVAL_FILE = "extraction_eval.jsonl"`, a held-out set the model never trained on. It does **not** exist yet; a dedicated evaluation set (and an expanded training set) will be generated separately. Until then this cell will raise a *file not found* error — that is expected.
>
> Generation here is **greedy** (`do_sample=False`) to mirror the `temp=0` runtime, so eval-time behaviour matches production. The current numbers reflect the **merged 16-bit** model; the deployed **Q4_K_M GGUF** should also be evaluated later, since quantization can shift outputs.

In [ ]:
!pip install -q sacrebleu sentence-transformers

import re
import json
import string
from collections import Counter

import torch
import sacrebleu
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer

EVAL_FILE = "extraction_eval.jsonl"   # <-- held-out split, NOT the training file
MAX_NEW_TOKENS = 128

# ---- 1. Deterministic generation (mirrors runtime Temp=0.0) ----
FastLanguageModel.for_inference(model)   # Unsloth 2x faster inference path

def extract_claim(instruction: str, input_text: str) -> str:
    """Generate one normalized claim greedily (reproducible)."""
    prompt = alpaca_prompt.format(
        instruction=instruction, input=input_text, output=""
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,                      # greedy = deterministic
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    return decoded.split("### Response:")[-1].strip()

# ---- 2. Surface-level metrics (pure Python, no hidden deps) ----
def normalize(s: str) -> str:
    s = s.lower().strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", s).strip()

def token_f1(pred: str, ref: str) -> float:
    p, r = normalize(pred).split(), normalize(ref).split()
    if not p or not r:
        return float(p == r)
    common = Counter(p) & Counter(r)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(r)
    return 2 * precision * recall / (precision + recall)

def rouge_l(pred: str, ref: str) -> float:
    p, r = normalize(pred).split(), normalize(ref).split()
    if not p or not r:
        return float(p == r)
    # LCS length via DP
    dp = [[0] * (len(r) + 1) for _ in range(len(p) + 1)]
    for i in range(1, len(p) + 1):
        for j in range(1, len(r) + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if p[i-1] == r[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[len(p)][len(r)]
    precision, recall = lcs / len(p), lcs / len(r)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# ---- 3. Run inference over the held-out set ----
rows = [json.loads(l) for l in open(EVAL_FILE, encoding="utf-8") if l.strip()]
preds, refs, styles = [], [], []
for row in rows:
    pred = extract_claim(row["instruction"], row["input"])
    preds.append(pred)
    refs.append(row["output"])
    styles.append(row.get("language") or row.get("style") or "all")

# ---- 4. Semantic similarity (reuses your pipeline embedder) ----
sem_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
emb_p = sem_model.encode(preds, convert_to_tensor=True, normalize_embeddings=True)
emb_r = sem_model.encode(refs,  convert_to_tensor=True, normalize_embeddings=True)
cos = (emb_p * emb_r).sum(dim=1).cpu().tolist()   # normalized -> dot = cosine

# ---- 5. Aggregate, overall + per language style ----
def report(idxs, label):
    n = len(idxs)
    em   = sum(normalize(preds[i]) == normalize(refs[i]) for i in idxs) / n
    f1   = sum(token_f1(preds[i], refs[i]) for i in idxs) / n
    rl   = sum(rouge_l(preds[i], refs[i]) for i in idxs) / n
    chrf = sacrebleu.corpus_chrf([preds[i] for i in idxs],
                                 [[refs[i] for i in idxs]]).score
    sem  = sum(cos[i] for i in idxs) / n
    print(f"{label:<16} n={n:<4} EM={em:.3f}  F1={f1:.3f}  "
          f"ROUGE-L={rl:.3f}  chrF={chrf:5.1f}  SemCos={sem:.3f}")

print(f"{'STYLE':<16} {'COUNT':<6} {'METRICS'}")
report(list(range(len(rows))), "OVERALL")
for s in sorted(set(styles)):
    if s != "all":
        report([i for i, st in enumerate(styles) if st == s], s)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

STYLE            COUNT  METRICS
OVERALL          n=26   EM=0.000  F1=0.055  ROUGE-L=0.045  chrF= 16.1  SemCos=0.035
